In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib
import scipy.io as sio
import copy
from scipy.stats import norm
from matplotlib import animation, rc
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.patches import Circle
from matplotlib.patches import Arrow
from matplotlib.collections import PatchCollection
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import sys
# for the font of the plots
matplotlib.rc("font", **{"family":"serif","serif":["Computer Modern Roman"]})
matplotlib.rc("text", usetex=True)

# load data

In [ ]:
numx = 60
numy = 30
N = numx*numy
nrec = 1000

base_dir = 'directory_of_data'

file_name = 'xarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    xarr = np.fromfile(f,np.float64,sep=' ')
xarr = xarr.reshape((nrec,N))

file_name = 'yarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    yarr = np.fromfile(f,np.float64,sep=' ')
yarr = yarr.reshape((nrec,N))

file_name = 'vxarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    vxarr = np.fromfile(f,np.float64,sep=' ')
vxarr = vxarr.reshape((nrec,N))

file_name = 'vyarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    vyarr = np.fromfile(f,np.float64,sep=' ')
vyarr = vyarr.reshape((nrec,N))


# snapshot

In [ ]:
numx = 60
numy = 30
R = 0.5
spacing = 1.0
dx = 0.4*spacing
dy = 0.4*spacing
Lx = spacing * 2 * R * numx
Ly = 0.5 * np.sqrt(3) * numy * spacing * 2 * R

thiscm = 'twilight_shifted'

ri = np.sqrt((xarr-0.5*Lx)**2 + (yarr-0.5*Ly)**2)
th_lf = np.arctan2(yarr-0.5*Ly,xarr-0.5*Lx)

tlen = np.size(ri,axis=0)
thistime = 999 # last time step
w_test = 0.00006 # global rotational rate obtained by trial-and-error

# move to rotational frame
x_mov = ri[thistime,:]*np.cos(th_lf[thistime,:]-thistime*w_test) + 0.5*Lx
y_mov = ri[thistime,:]*np.sin(th_lf[thistime,:]-thistime*w_test) + 0.5*Ly
x_prev = ri[thistime-1,:]*np.cos(th_lf[thistime-1,:]-(thistime-1)*w_test) + 0.5*Lx
y_prev = ri[thistime-1,:]*np.sin(th_lf[thistime-1,:]-(thistime-1)*w_test) + 0.5*Ly
vx_mov = x_mov-x_prev
vy_mov = y_mov-y_prev
th_mov = np.arctan2(vy_mov,vx_mov) # angle of velocity vector

%matplotlib inline
fig,ax = plt.subplots(figsize=(20,6),facecolor=(1,1,1))

ax.plot(x_mov,y_mov,'ko',markersize=2)
for i in range(N):
    embryo = plt.Circle((x_mov[i],y_mov[i]),R,fill=False)
    plt.gcf().gca().add_artist(embryo)
    
ax.set_xlim(-2,Lx+2)
ax.set_ylim(-4,Ly+4)
ax.set_aspect('equal')
ax.tick_params(labelsize=15)

thetanorm = plt.Normalize(vmin=-np.pi, vmax=np.pi)
img = ax.scatter(x_mov,y_mov,s=80,c=th_mov,cmap=thiscm,norm=thetanorm) # color code by velocity vector angle
cb = fig.colorbar(img,ax=ax)
cb.set_label('$\\theta$',fontsize=20)
cb.ax.tick_params(labelsize=15)

plt.show()
#fig.savefig('config_laststep_cos_abs_gauss_halfq_rotFR_v3.pdf',dpi=fig.dpi,bbox_inches='tight')


# movie

In [ ]:
numx = 60
numy = 30
R = 0.5
spacing = 1.0
dx = 0.4*spacing
dy = 0.4*spacing
Lx = spacing * 2 * R * numx
Ly = 0.5 * np.sqrt(3) * numy * spacing * 2 * R

thiscm = 'twilight_shifted'

# make movie in the rotational frame 
# from the 500th index because it takes long time to generate a movie. Can choose any starting point.
ri = np.sqrt((xarr[-500:,:]-0.5*Lx)**2 + (yarr[-500:,:]-0.5*Ly)**2)
th_lf = np.arctan2(yarr[-500:,:]-0.5*Ly,xarr[-500:,:]-0.5*Lx)

tlen = np.size(ri,axis=0)
thistime = 1
w_test = 0.00006

x_mov = ri[thistime,:]*np.cos(th_lf[thistime,:]-thistime*w_test) + 0.5*Lx
y_mov = ri[thistime,:]*np.sin(th_lf[thistime,:]-thistime*w_test) + 0.5*Ly
x_prev = ri[thistime-1,:]*np.cos(th_lf[thistime-1,:]-(thistime-1)*w_test) + 0.5*Lx
y_prev = ri[thistime-1,:]*np.sin(th_lf[thistime-1,:]-(thistime-1)*w_test) + 0.5*Ly
vx_mov = x_mov-x_prev
vy_mov = y_mov-y_prev
th_mov = np.arctan2(vy_mov,vx_mov)

%matplotlib inline
fig,ax = plt.subplots(figsize=(16,8),facecolor=(1,1,1))

ax.plot(x_mov,y_mov,'ko',markersize=2)
for i in range(N):
    embryo = plt.Circle((x_mov[i],y_mov[i]),R,fill=False)
    plt.gcf().gca().add_artist(embryo)
    
ax.set_xlim(-2,Lx+2)
ax.set_ylim(-4,Ly+4)
ax.tick_params(labelsize=15)

thetanorm = plt.Normalize(vmin=-np.pi, vmax=np.pi)
img = ax.scatter(x_mov,y_mov,s=80,c=th_mov,cmap=thiscm,norm=thetanorm)
cb = fig.colorbar(img,ax=ax,shrink=0.8)
cb.set_label('$\\theta$',fontsize=20)
cb.ax.tick_params(labelsize=15)

def animate(s):
    ax.clear()
    
    x_mov = ri[s+1,:]*np.cos(th_lf[s+1,:]-(s+1)*w_test) + 0.5*Lx
    y_mov = ri[s+1,:]*np.sin(th_lf[s+1,:]-(s+1)*w_test) + 0.5*Ly
    x_prev = ri[s,:]*np.cos(th_lf[s,:]-(s)*w_test) + 0.5*Lx
    y_prev = ri[s,:]*np.sin(th_lf[s,:]-(s)*w_test) + 0.5*Ly
    vx_mov = x_mov-x_prev
    vy_mov = y_mov-y_prev
    th_mov = np.arctan2(vy_mov,vx_mov)
    
    ax.plot(x_mov,y_mov,'ko',markersize=2)
    
    for i in range(N):
        embryo = plt.Circle((x_mov[i],y_mov[i]),R,fill=False)
        plt.gcf().gca().add_artist(embryo)
    
    ax.set_xlim(-2,Lx+2)
    ax.set_ylim(-4,Ly+4)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=15)
    
    img = ax.scatter(x_mov,y_mov,s=80,c=th_mov,cmap=thiscm,norm=thetanorm)
    
anim = FuncAnimation(fig,animate,frames=tlen-1,interval=30,blit=False)
HTML(anim.to_html5_video())

writervideo = animation.FFMpegWriter(fps=10)
anim.save('mv_propag.mp4',writer=writervideo)


# dispersion relation from multiple runs

In [ ]:
# simulation parameters
# toy model

numx = 60
numy = 30
N = numx*numy
R = 0.5

spacing_tm = 1.0
dx_tm = 0.4*spacing_tm
dy_tm = 0.4*spacing_tm
Lx_tm = spacing_tm * 2 * R * numx * 3
Ly_tm = 0.5 * np.sqrt(3) * numy * spacing_tm * 2 * R * 3
xnum_tm = round(Lx_tm/dx_tm)
ynum_tm = round(Ly_tm/dy_tm)

nrec = 1000
wnum = int(nrec/2)+1

dkx_tm = 2*np.pi/Lx_tm
dky_tm = 2*np.pi/Ly_tm

dt_tm = 0.4
dw_tm = 2*np.pi/(dt_tm*nrec)


# load data

In [ ]:
base_dir = 'directory_of_data'

file_name = 'Cll_toy.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_tm = np.fromfile(f,np.float64)
Cll_tm = (1/N)*Cll_tm.reshape((xnum_tm,ynum_tm,wnum))

file_name = 'Ctt_toy.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_tm = np.fromfile(f,np.float64)
Ctt_tm = (1/N)*Ctt_tm.reshape((xnum_tm,ynum_tm,wnum))


In [ ]:
# frequency domain 
# toy model

kxrange_tm = np.zeros(xnum_tm)
kyrange_tm = np.zeros(ynum_tm)
wrange_tm = dw_tm*np.arange(wnum)

for i in range(xnum_tm):
    kxrange_tm[i] = -np.pi/dx_tm + dkx_tm*i
for j in range(ynum_tm):
    kyrange_tm[j] = -np.pi/dy_tm + dky_tm*j
    
kx_tm,ky_tm = np.meshgrid(kxrange_tm,kyrange_tm,indexing='ij')

In [ ]:
# 2d matrix whose elements are the max value of C at particular (qx,qy)
# preserve the sign of Cltr and Clti

Clltm_copy = copy.copy(Cll_tm)
Ctttm_copy = copy.copy(Ctt_tm)
Cfulltm_copy = Clltm_copy + Ctttm_copy

Clltm_peak = np.zeros([xnum_tm,ynum_tm])
Ctttm_peak = np.zeros([xnum_tm,ynum_tm])
Cfulltm_peak = np.zeros([xnum_tm,ynum_tm])

wlltm_peak = np.zeros([xnum_tm,ynum_tm])
wtttm_peak = np.zeros([xnum_tm,ynum_tm])
wfulltm_peak = np.zeros([xnum_tm,ynum_tm])

fromhere = 10

for i in range(xnum_tm):
    for j in range(ynum_tm):
        Cll_thisq = Clltm_copy[i,j,fromhere:]
        Ctt_thisq = Ctttm_copy[i,j,fromhere:]
        Cfull_thisq = Cfulltm_copy[i,j,fromhere:]
        
        Clltm_peak[i,j] = np.max(Cll_thisq) # record max C value at certain q
        idx_wllmax = np.where(Cll_thisq==np.max(Cll_thisq)) # index in w dimension for the max C
        idx_wllmax = int(idx_wllmax[0][0]) + fromhere # make the index an integer and add fromhere
        wlltm_peak[i,j] = wrange_tm[idx_wllmax] # record the value of max peak w. 
        
        Ctttm_peak[i,j] = np.max(Ctt_thisq) # record max C value at certain q
        idx_wttmax = np.where(Ctt_thisq==np.max(Ctt_thisq)) # index in w dimension for the max C
        idx_wttmax = int(idx_wttmax[0][0]) + fromhere # make the index an integer and add fromhere
        wtttm_peak[i,j] = wrange_tm[idx_wttmax] # record the value of max peak w. 
        
        Cfulltm_peak[i,j] = np.max(Cfull_thisq)
        idx_wfullmax = np.where(Cfull_thisq==np.max(Cfull_thisq))
        idx_wfullmax = int(idx_wfullmax[0][0]) + fromhere
        wfulltm_peak[i,j] = wrange_tm[idx_wfullmax]

In [ ]:
# points for reciprocal lattice and the first BZ

# simulation
p1x_tm = 0.0 # kx value of the vertex of reciprocal lattice obtained from previous result
p1y_tm = 4*np.pi/(np.sqrt(3)*spacing_tm)

p2x_tm = (1/2)*p1x_tm-(np.sqrt(3)/2)*p1y_tm
p2y_tm = (np.sqrt(3)/2)*p1x_tm+(1/2)*p1y_tm

p3x_tm = (1/2)*p2x_tm-(np.sqrt(3)/2)*p2y_tm
p3y_tm = (np.sqrt(3)/2)*p2x_tm+(1/2)*p2y_tm

p4x_tm = (1/2)*p3x_tm-(np.sqrt(3)/2)*p3y_tm
p4y_tm = (np.sqrt(3)/2)*p3x_tm+(1/2)*p3y_tm

p5x_tm = (1/2)*p4x_tm-(np.sqrt(3)/2)*p4y_tm
p5y_tm = (np.sqrt(3)/2)*p4x_tm+(1/2)*p4y_tm

p6x_tm = (1/2)*p5x_tm-(np.sqrt(3)/2)*p5y_tm
p6y_tm = (np.sqrt(3)/2)*p5x_tm+(1/2)*p5y_tm

bz1x_tm = (1/2)*p1x_tm + (1/(2*np.sqrt(3)))*p1y_tm
bz1y_tm = -(1/(2*np.sqrt(3)))*p1x_tm + (1/2)*p1y_tm

bz2x_tm = (1/2)*bz1x_tm-(np.sqrt(3)/2)*bz1y_tm
bz2y_tm = (np.sqrt(3)/2)*bz1x_tm+(1/2)*bz1y_tm

bz3x_tm = (1/2)*bz2x_tm-(np.sqrt(3)/2)*bz2y_tm
bz3y_tm = (np.sqrt(3)/2)*bz2x_tm+(1/2)*bz2y_tm

bz4x_tm = (1/2)*bz3x_tm-(np.sqrt(3)/2)*bz3y_tm
bz4y_tm = (np.sqrt(3)/2)*bz3x_tm+(1/2)*bz3y_tm

bz5x_tm = (1/2)*bz4x_tm-(np.sqrt(3)/2)*bz4y_tm
bz5y_tm = (np.sqrt(3)/2)*bz4x_tm+(1/2)*bz4y_tm

bz6x_tm = (1/2)*bz5x_tm-(np.sqrt(3)/2)*bz5y_tm
bz6y_tm = (np.sqrt(3)/2)*bz5x_tm+(1/2)*bz5y_tm


In [ ]:
# pbc
#Gpoint = np.array([74,32],dtype=int)
#Mpoint = np.array([104,40],dtype=int)
#Kpoint = np.array([114,32],dtype=int)

# fbc box size 2L
#Gpoint = np.array([148,65],dtype=int)
#Mpoint = np.array([208,80],dtype=int)
#Kpoint = np.array([228,64],dtype=int)
#Kpoint = np.array([69,65],dtype=int)
#Mpoint = np.array([89,80],dtype=int)

# fbc box size 3L
Gpoint = np.array([222,96],dtype=int)
Mpoint = np.array([312,120],dtype=int)
Kpoint = np.array([342,96],dtype=int)


# 3d dispersion

In [ ]:
Ctm_plot = copy.copy(Cfulltm_peak)
wtm_plot = copy.copy(wfulltm_peak)

peak_thres = 0.0

kxtm_thres = kx_tm[np.abs(Ctm_plot)>peak_thres]
kytm_thres = ky_tm[np.abs(Ctm_plot)>peak_thres]
wpktm_thres = wtm_plot[np.abs(Ctm_plot)>peak_thres]

#%matplotlib notebook
%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(15,13))
ax = fig.add_subplot(111,projection='3d')

img = ax.scatter3D(kxtm_thres,kytm_thres,wpktm_thres,c=Ctm_plot[Ctm_plot>peak_thres],marker='.',cmap = 'viridis',rasterized=True)

cb = ax.figure.colorbar(img,shrink=0.7,pad=0.1)
ax.set_xlabel('$q_x$', fontsize=20, labelpad=10)
ax.set_ylabel('$q_y$', fontsize=20, labelpad=10)
ax.set_zlabel('$\omega$', fontsize=20, labelpad=10)

cb.set_label('C',fontsize=20)
ax.tick_params(labelsize=15)
cb.ax.tick_params(labelsize=15)
#plt.savefig('expt_dispersion_v6_thres001.pdf',dpi=600,bbox_inches='tight')

plt.show()


# dispersion along a path on the 1st Brillouin zone

In [ ]:
# indices for the path G->K->M->G

idisGK = Kpoint[0]-Gpoint[0]
jdisGK = Kpoint[1]-Gpoint[1]
numGtoK = max(np.abs(idisGK),np.abs(jdisGK))
GtoK = np.zeros([2,numGtoK],dtype = int)
GtoK[0,0]=Gpoint[0]
GtoK[1,0]=Gpoint[1]
if np.abs(idisGK)>np.abs(jdisGK):
    idisGKsign = idisGK/np.abs(idisGK)
    for i in range(1,numGtoK):
        GtoK[0,i]=GtoK[0,0]+i*idisGKsign
        GtoK[1,i]=int(GtoK[1,0]+i*idisGKsign*jdisGK/idisGK)
else:
    jdisGKsign = jdisGK/np.abs(jdisGK)
    for i in range(1,numGtoK):
        GtoK[1,i]=GtoK[1,0]+i*jdisGKsign
        GtoK[0,i]=int(GtoK[0,0]+i*jdisGKsign*idisGK/jdisGK)

idisKM = Mpoint[0]-Kpoint[0]
jdisKM = Mpoint[1]-Kpoint[1]
numKtoM = max(np.abs(idisKM),np.abs(jdisKM))
KtoM = np.zeros([2,numKtoM],dtype = int)
KtoM[0,0]=Kpoint[0]
KtoM[1,0]=Kpoint[1]
if np.abs(idisKM)>np.abs(jdisKM):
    idisKMsign = idisKM/np.abs(idisKM)
    for i in range(1,numKtoM):
        KtoM[0,i]=KtoM[0,0]+i*idisKMsign
        KtoM[1,i]=int(KtoM[1,0]+i*idisKMsign*jdisKM/idisKM)
else:
    jdisKMsign = jdisKM/np.abs(jdisKM)
    for i in range(1,numKtoM):
        KtoM[1,i]=KtoM[1,0]+i*jdisKMsign
        KtoM[0,i]=int(KtoM[0,0]+i*jdisKMsign*idisKM/jdisKM)
        
idisMG = Gpoint[0]-Mpoint[0]
jdisMG = Gpoint[1]-Mpoint[1]
numMtoG = max(np.abs(idisMG),np.abs(jdisMG))
MtoG = np.zeros([2,numMtoG+1],dtype = int)
MtoG[0,0]=Mpoint[0]
MtoG[1,0]=Mpoint[1]
if np.abs(idisMG)>np.abs(jdisMG):
    idisMGsign = idisMG/np.abs(idisMG)
    for i in range(1,numMtoG):
        MtoG[0,i]=MtoG[0,0]+i*idisMGsign
        MtoG[1,i]=int(MtoG[1,0]+i*idisMGsign*jdisMG/idisMG)
else:
    jdisMGsign = jdisMG/np.abs(jdisMG)
    for i in range(1,numMtoG):
        MtoG[1,i]=MtoG[1,0]+i*jdisMGsign
        MtoG[0,i]=int(MtoG[0,0]+i*jdisMGsign*idisMG/jdisMG)
MtoG[0,-1]=Gpoint[0]
MtoG[1,-1]=Gpoint[1]

In [ ]:
Clltm_GtoK = copy.copy(Cll_tm)
Clltm_GtoK = Clltm_GtoK[GtoK[0,:],GtoK[1,:],:]
Clltm_KtoM = copy.copy(Cll_tm)
Clltm_KtoM = Clltm_KtoM[KtoM[0,:],KtoM[1,:],:]
Clltm_MtoG = copy.copy(Cll_tm)
Clltm_MtoG = Clltm_MtoG[MtoG[0,:],MtoG[1,:],:]
Clltm_GKMG = np.concatenate((Clltm_GtoK,Clltm_KtoM,Clltm_MtoG),axis=0)

Ctttm_GtoK = copy.copy(Ctt_tm)
Ctttm_GtoK = Ctttm_GtoK[GtoK[0,:],GtoK[1,:],:]
Ctttm_KtoM = copy.copy(Ctt_tm)
Ctttm_KtoM = Ctttm_KtoM[KtoM[0,:],KtoM[1,:],:]
Ctttm_MtoG = copy.copy(Ctt_tm)
Ctttm_MtoG = Ctttm_MtoG[MtoG[0,:],MtoG[1,:],:]
Ctttm_GKMG = np.concatenate((Ctttm_GtoK,Ctttm_KtoM,Ctttm_MtoG),axis=0)

In [ ]:
qnum_tm = np.size(Clltm_GKMG,axis=0)
qtm_band,wtm_band = np.meshgrid(np.arange(qnum_tm),wrange_tm,indexing='ij')

fromthisw = 10 # not include w=0
uptothisw = -1 # high frequency w is not relevant
wtm_band = wtm_band[:,fromthisw:uptothisw] # only take w>0.
qtm_band = qtm_band[:,fromthisw:uptothisw]
Clltm_band = Clltm_GKMG[:,fromthisw:uptothisw]
Ctttm_band = Ctttm_GKMG[:,fromthisw:uptothisw]


In [ ]:
tickloc = [0,np.size(GtoK,axis=1),np.size(GtoK,axis=1)+np.size(KtoM,axis=1)-1,np.size(GtoK,axis=1)+np.size(KtoM,axis=1)+np.size(MtoG,axis=1)-1]

Ctm_band = copy.copy(Clltm_band)+copy.copy(Ctttm_band)

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(10,8))
img = plt.contourf(qtm_band,wtm_band,Ctm_band,levels=30)

cb = fig.colorbar(img)
plt.xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=20)
plt.ylabel('$\omega$',fontsize=20)
cb.set_label('C',fontsize=20)

plt.show()


# plot all together

In [ ]:
tickloc = [0,np.size(GtoK,axis=1),np.size(GtoK,axis=1)+np.size(KtoM,axis=1)-1,np.size(GtoK,axis=1)+np.size(KtoM,axis=1)+np.size(MtoG,axis=1)-1]

Ctm_band = copy.copy(Clltm_band)+copy.copy(Ctttm_band)

Ctm_plot = copy.copy(Cfulltm_peak)
wtm_plot = copy.copy(wfulltm_peak)

peak_thres = 0.02

kxtm_thres = kx_tm[np.abs(Ctm_plot)>peak_thres]
kytm_thres = ky_tm[np.abs(Ctm_plot)>peak_thres]
wpktm_thres = wtm_plot[np.abs(Ctm_plot)>peak_thres]


%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(20,7))
ax1 = fig.add_subplot(1,2,1,projection='3d')
ax2 = fig.add_subplot(1,2,2)

# first plot
img1 = ax1.scatter3D(kxtm_thres,kytm_thres,wpktm_thres,c=Ctm_plot[Ctm_plot>peak_thres],marker='.',cmap = 'viridis',rasterized=True)

ax1.plot([p1x_tm,p2x_tm,p3x_tm,p4x_tm,p5x_tm,p6x_tm,p1x_tm],[p1y_tm,p2y_tm,p3y_tm,p4y_tm,p5y_tm,p6y_tm,p1y_tm],zs=0,zdir='z',color='k')
ax1.plot([bz1x_tm,bz2x_tm,bz3x_tm,bz4x_tm,bz5x_tm,bz6x_tm,bz1x_tm],[bz1y_tm,bz2y_tm,bz3y_tm,bz4y_tm,bz5y_tm,bz6y_tm,bz1y_tm],zs=0,zdir='z',color='k')
ax1.plot([kx_tm[Gpoint[0],Gpoint[1]],kx_tm[Kpoint[0],Kpoint[1]],kx_tm[Mpoint[0],Mpoint[1]],kx_tm[Gpoint[0],Gpoint[1]]],[ky_tm[Gpoint[0],Gpoint[1]],ky_tm[Kpoint[0],Kpoint[1]],ky_tm[Mpoint[0],Mpoint[1]],ky_tm[Gpoint[0],Gpoint[1]]],zs=0,zdir='z',color='r',linestyle='--')

cb1 = ax1.figure.colorbar(img1,shrink=0.7,pad=0.1)
ax1.set_xlabel('$q_x$', fontsize=40, labelpad=10)
ax1.set_ylabel('$q_y$', fontsize=40, labelpad=10)
ax1.set_zlabel('$\omega$', fontsize=40, labelpad=10)
ax1.text2D(-0.065,0.088,'(a)',fontsize=40,va='top',ha='right')
cb1.set_label('C',fontsize=40)
ax1.tick_params(labelsize=20)
cb1.ax.tick_params(labelsize=20)

# second plot
img2 = ax2.contourf(qtm_band,wtm_band,Ctm_band,levels=100)

cb2 = fig.colorbar(img2,ax=ax2)
ax2.set_ylabel('$\omega$',fontsize=40)
cb2.set_label('C',fontsize=40)
ax2.text(-0.8,7.8,'(b)',fontsize=40,va='top',ha='right')
ax2.tick_params(labelsize=20)
cb2.ax.tick_params(labelsize=20)
ax2.set_xticks(tickloc,["$\Gamma$","K","M","$\Gamma$"],fontsize=40)

# to avoid the weird white lines appearing in pdf file of the plot
for c in img2.collections:
    c.set_edgecolor("face")

fig.tight_layout()
plt.show()
#fig.savefig('disp_result_v2.pdf',dpi=fig.dpi,bbox_inches='tight')
